# PDHG CT Batched Tuning In Colab

This notebook keeps Colab as a thin launcher around the repo's batched PDHG tuning runner for `transmission_ct`.

It will:
- mount Google Drive
- fetch the repo into `/content`
- install only the CT tuning dependencies
- optionally point the run at a DICOM folder in Drive
- write a Colab-specific CT tuning config
- dry-run the grid
- launch the batched sweep
- track `progress.json`
- show the latest leaderboard


## 1. Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

If available, prefer `A100`. This notebook is designed for one-image batched candidate sweeps where chunk size controls the amount of parallelism.

In [ ]:
# Project settings

SETUP_MODE = "git"  # "git" or "drive_zip"
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"
REPO_BRANCH = "main"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"

REPO_DIR = "/content/mycode2"
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_ct_tuning_exports"
DRIVE_CT_DATA_DIR = ""  # e.g. /content/drive/MyDrive/.../Low Dose Images ...
DRIVE_CHECKPOINT_DIR = ""  # optional local folder containing lodochallenge_pixel_diffusion
HF_MODEL_ID = "jiayangshi/lodochallenge_pixel_diffusion"

SESSION_TAG = "ct"
CANDIDATE_CHUNK_SIZE = 8
MAX_CANDIDATES = 0
PROGRESS_UPDATE_EVERY = 10

TOTAL_IMAGES = 1
BATCH_SIZE = 1
DATA_START_IDX = 0
DATA_END_IDX = 1

NUM_STEPS = 400
MAX_ITER = 400

SIGMA_MAX_VALUES = [20, 27, 30]
SIGMA_MIN_VALUES = [0.05, 0.075, 0.1]
TAU_VALUES = [0.005, 0.0075, 0.01]
SIGMA_DUAL_VALUES = [800, 1200, 1600]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    if not REPO_URL:
        raise ValueError("Fill in REPO_URL before running this cell.")
    !git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as handle:
        handle.extractall("/content")
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

%cd /content/mycode2
!git status --short

In [ ]:
%cd /content/mycode2
!pip install -r requirements-colab-ct.txt

In [ ]:
# Optional: copy a local DM4CT checkpoint folder from Drive into the repo.
from pathlib import Path
import shutil

repo_dir = Path(REPO_DIR)
if DRIVE_CHECKPOINT_DIR:
    src = Path(DRIVE_CHECKPOINT_DIR)
    dst = repo_dir / "pretrained-models" / "dm4ct" / "lodochallenge_pixel_diffusion"
    if not src.exists():
        raise FileNotFoundError(f"Checkpoint dir not found: {src}")
    if dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print(f"Copied checkpoint to {dst}")
else:
    print("Skipping checkpoint copy. The notebook will use Hugging Face by default.")

## 2. Write A Colab-Specific CT Tuning Config

In [ ]:
from pathlib import Path
import yaml

repo_dir = Path(REPO_DIR)
template_path = repo_dir / "tuning" / "pdhg_batched_grid_ct.template.yaml"
custom_path = repo_dir / f"tuning/pdhg_batched_grid_ct.colab_{SESSION_TAG}.yaml"

with template_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["study"]["name"] = f"pdhg_transmission_ct_colab_{SESSION_TAG}"
cfg["batched"]["candidate_chunk_size"] = int(CANDIDATE_CHUNK_SIZE)
cfg["batched"]["progress_update_every"] = int(PROGRESS_UPDATE_EVERY)

base_overrides = [item for item in cfg.get("base_overrides", []) if not item.startswith("total_images=")]
base_overrides = [item for item in base_overrides if not item.startswith("batch_size=")]
base_overrides = [item for item in base_overrides if not item.startswith("sampler.annealing_scheduler_config.num_steps=")]
base_overrides = [item for item in base_overrides if not item.startswith("inverse_task.admm_config.max_iter=")]
base_overrides.extend([
    f"total_images={int(TOTAL_IMAGES)}",
    f"batch_size={int(BATCH_SIZE)}",
    f"sampler.annealing_scheduler_config.num_steps={int(NUM_STEPS)}",
    f"inverse_task.admm_config.max_iter={int(MAX_ITER)}",
    f"data.start_idx={int(DATA_START_IDX)}",
    f"data.end_idx={int(DATA_END_IDX)}",
])
if DRIVE_CT_DATA_DIR:
    base_overrides = [item for item in base_overrides if not item.startswith("data.image_root_path=")]
    base_overrides.append(f"data.image_root_path={DRIVE_CT_DATA_DIR}")
cfg["base_overrides"] = base_overrides

base_overrides = [item for item in base_overrides if not item.startswith("model.model_config.model_id=")]
base_overrides = [item for item in base_overrides if not item.startswith("model.model_config.local_files_only=")]
if DRIVE_CHECKPOINT_DIR:
    base_overrides.extend([
        "model.model_config.model_id=pretrained-models/dm4ct/lodochallenge_pixel_diffusion",
        "model.model_config.local_files_only=true",
    ])
else:
    base_overrides.extend([
        f"model.model_config.model_id={HF_MODEL_ID}",
        "model.model_config.local_files_only=false",
    ])
cfg["base_overrides"] = base_overrides

cfg["grid"] = {
    "sampler.annealing_scheduler_config.sigma_max": {"values": list(SIGMA_MAX_VALUES)},
    "sampler.annealing_scheduler_config.sigma_min": {"values": list(SIGMA_MIN_VALUES)},
    "inverse_task.admm_config.pdhg.tau": {"values": list(TAU_VALUES)},
    "inverse_task.admm_config.pdhg.sigma_dual": {"values": list(SIGMA_DUAL_VALUES)},
}

with custom_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

CUSTOM_TUNING_CONFIG = str(custom_path.relative_to(repo_dir).as_posix())
print(f"Wrote {custom_path}")
print(custom_path.read_text(encoding="utf-8"))

In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_batched_grid.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
    "--dry-run",
]
if MAX_CANDIDATES > 0:
    cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, cwd=repo_dir, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, cmd)

In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_batched_grid.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
]
if MAX_CANDIDATES > 0:
    cmd.extend(["--max-candidates", str(MAX_CANDIDATES)])

log_path = repo_dir / "tuning_runs" / f"latest_ct_colab_run_{SESSION_TAG}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("w", encoding="utf-8") as handle:
    proc = subprocess.Popen(
        cmd,
        cwd=repo_dir,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

pid_path = repo_dir / "tuning_runs" / f"latest_ct_colab_run_{SESSION_TAG}.pid"
pid_path.write_text(str(proc.pid), encoding="utf-8")

print("Launched in background:")
print(" ".join(shlex.quote(part) for part in cmd))
print(f"PID: {proc.pid}")
print(f"Log: {log_path}")

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / f"pdhg_transmission_ct_colab_{SESSION_TAG}"
if not study_root.exists() or not any(study_root.iterdir()):
    raise FileNotFoundError(f"No study runs found at {study_root}")

runs = sorted([path for path in study_root.iterdir() if path.is_dir()])
latest_study = runs[-1]
progress_path = latest_study / "progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8"))

print(f"Study: {latest_study}")
print(f"Status: {progress['status']}")
print(f"Updated: {progress['updated_at']}")
print(f"Chunks: {progress['completed_chunks']}/{progress['total_chunks']} | Candidates: {progress['completed_candidates']}/{progress['total_candidates']}")
print(f"Elapsed: {progress['elapsed_seconds'] / 60:.1f} min")
eta_seconds = progress.get('eta_seconds')
print('ETA: n/a' if eta_seconds is None else f"ETA: {eta_seconds / 60:.1f} min")

best = progress.get("best_so_far")
if best:
    print("Current best:")
    print(f"  idx={best['candidate_index']} score={best['score']:.4f} sigma_max={best['sigma_max']} sigma_min={best['sigma_min']} tau={best['tau']} sigma_dual={best['sigma_dual']}")

best_preview_path = latest_study / "best_preview.png"
if best_preview_path.exists():
    display(Image(filename=str(best_preview_path)))

In [ ]:
from pathlib import Path
import pandas as pd

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / f"pdhg_transmission_ct_colab_{SESSION_TAG}"
runs = sorted([path for path in study_root.iterdir() if path.is_dir()])
latest_study = runs[-1]
leaderboard = latest_study / "leaderboard.csv"
df = pd.read_csv(leaderboard)
display(df)

In [ ]:
from pathlib import Path
import shutil

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / f"pdhg_transmission_ct_colab_{SESSION_TAG}"
runs = sorted([path for path in study_root.iterdir() if path.is_dir()])
latest_study = runs[-1]

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
target = export_root / latest_study.name
if target.exists():
    shutil.rmtree(target)
shutil.copytree(latest_study, target)
print(f"Copied study summary to {target}")